# 01 Preprocess: Stages 1-7

Thin entry point that runs preprocessing, OCR, and text classification on a single SIP PDF. All real logic lives in the `sip_extractor` package; this notebook just wires together inputs, outputs, and inline previews.

The install cell picks the paddle build based on the runtime: `paddlepaddle-gpu==3.0.0` on a GPU runtime (T4/A100, wheel pulled from Paddle's own index since it is not on PyPI), `paddlepaddle==3.0.0` on a CPU runtime. Stages 1-5 take ~10-30s either way. Stage 6 (PaddleOCR) is ~5-10s per tile on T4 versus ~30-60s per tile on CPU; on the MUNDEWADI test SIP that's roughly ~2 minutes on T4 versus ~10+ minutes on CPU. Switch to T4 if you want OCR to be fast; CPU still works.

## Setup

Detects whether we are on Colab or local. On Colab, mounts Drive and points the model caches at `.model_cache/` so PaddleOCR / DINOv2 / Qwen don't redownload every session. Locally, `PROJECT_ROOT` is the repo root.

In [ ]:
import os
import sys
from pathlib import Path

# Disable paddle's oneDNN backend and PIR executor before any paddle import.
# Some paddleocr 3.5.0 + paddlepaddle CPU combos crash inside an unimplemented
# oneDNN attribute conversion (ConvertPirAttribute2RuntimeAttribute). Forcing
# the legacy executor and turning off oneDNN avoids that code path.
os.environ.setdefault("FLAGS_use_mkldnn", "0")
os.environ.setdefault("FLAGS_enable_pir_in_executor", "0")

IS_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/sip-extractor")
    cache = PROJECT_ROOT / ".model_cache"
    cache.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache / "huggingface")
    os.environ["TORCH_HOME"] = str(cache / "torch")
    os.environ["PADDLE_HOME"] = str(cache / "paddle")
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Working dir: {PROJECT_ROOT}")
print(f"Colab: {IS_COLAB}")

In [ ]:
# Colab-only: pull the latest repo state from GitHub.
#
# After `git pull` modifies the notebook file on Drive, the open Colab tab is
# still showing the cached pre-pull JSON. Close and reopen the notebook to
# pick up new cells. Already-loaded cells in the current session will use the
# updated .py modules on next import (preprocessing, ocr, classify).
if IS_COLAB:
    print("Pulling latest from GitHub...")
    !git -C {PROJECT_ROOT} pull --ff-only origin main 2>&1 | tail -5
    print()
    print("If the pull added or modified notebook cells, close and reopen this")
    print("notebook tab to see them. Existing .py module changes load on next import.")

In [ ]:
# Colab-only: install runtime deps. Takes ~60-90s on first run per VM.
#
# Picks paddle build by runtime:
# - GPU runtime (nvidia-smi present): paddlepaddle-gpu==3.0.0 from Paddle's
#   own index (cu126 wheels exist for cp310/cp311 linux_x86_64; not on PyPI).
# - CPU runtime: paddlepaddle==3.0.0 from PyPI.
# Both pinned to 3.0.0 to avoid a PIR/oneDNN inference bug under paddleocr 3.5.0.
#
# If paddle is already installed but built for the wrong device, this cell
# uninstalls it and reinstalls the right build in the same kernel. Re-import
# of paddle in the same kernel after swap is best-effort; if the post-install
# verify still reports the wrong CUDA flag, restart the runtime.
#
# CPU fallback: if the GPU install lands but paddle still reports CUDA build:
# False, this cell uninstalls paddlepaddle-gpu, reinstalls CPU paddle, and
# asks for a runtime restart so OCR still runs (slowly).
#
# langchain_text_splitters: paddleocr 3.5.0 -> paddlex unconditionally imports
# this even when retrieval pipelines are unused.
import importlib
import shutil
import subprocess


def _has_gpu() -> bool:
    if shutil.which("nvidia-smi") is None:
        return False
    try:
        subprocess.run(["nvidia-smi"], capture_output=True, check=True, timeout=5)
        return True
    except Exception:
        return False


def _module_present(mod: str) -> bool:
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


HAS_GPU = _has_gpu()
PADDLE_PKG = "paddlepaddle-gpu==3.0.0" if HAS_GPU else "paddlepaddle==3.0.0"
PADDLE_INDEX = "https://www.paddlepaddle.org.cn/packages/stable/cu126/" if HAS_GPU else None

REQUIRED = {
    "fitz": "pymupdf",
    "cv2": "opencv-python",
    "skimage": "scikit-image",
    "paddleocr": "paddleocr==3.5.0",
    "paddle": PADDLE_PKG,
    "langchain_text_splitters": "langchain-text-splitters",
}

# Detect mismatch: paddle importable but built for the wrong device. After
# uninstall, sys.modules still has paddle cached, so _module_present would
# lie; force reinstall instead of relying on the import check.
force_paddle_install = False
if _module_present("paddle"):
    import paddle as _p
    _has_cuda = _p.is_compiled_with_cuda()
    if HAS_GPU and not _has_cuda:
        print("Installed paddle is CPU build but runtime has GPU. Swapping...")
        !pip uninstall -y paddlepaddle paddlepaddle-gpu 2>&1 | tail -3
        force_paddle_install = True
    elif (not HAS_GPU) and _has_cuda:
        print("Installed paddle is GPU build but runtime is CPU. Swapping...")
        !pip uninstall -y paddlepaddle paddlepaddle-gpu 2>&1 | tail -3
        force_paddle_install = True
    else:
        REQUIRED.pop("paddle", None)

missing = [pkg for mod, pkg in REQUIRED.items() if not _module_present(mod)]
if force_paddle_install and PADDLE_PKG not in missing:
    missing.append(PADDLE_PKG)

if IS_COLAB and missing:
    print(f"Runtime: {'GPU' if HAS_GPU else 'CPU'}  Installing: {missing}")
    if HAS_GPU and any("paddlepaddle-gpu" in m for m in missing):
        # paddlepaddle-gpu wheels are on Paddle's index, not PyPI.
        # --extra-index-url keeps PyPI available for transitive deps.
        gpu_pkgs = [m for m in missing if "paddlepaddle-gpu" in m]
        other_pkgs = [m for m in missing if "paddlepaddle-gpu" not in m]
        %pip install -q --index-url {PADDLE_INDEX} --extra-index-url https://pypi.org/simple {" ".join(gpu_pkgs)}
        if other_pkgs:
            %pip install -q {" ".join(other_pkgs)}
    else:
        %pip install -q {" ".join(missing)}
else:
    print(f"All deps present (IS_COLAB={IS_COLAB}, GPU={HAS_GPU})")

# Verify and print resolved state. If GPU runtime but paddle is still CPU
# build after install, fall back to CPU paddle so OCR still runs (slowly).
try:
    import paddle
    print(f"paddle {paddle.__version__}  CUDA build: {paddle.is_compiled_with_cuda()}  GPUs: {paddle.device.cuda.device_count() if paddle.is_compiled_with_cuda() else 0}")
    if HAS_GPU and not paddle.is_compiled_with_cuda():
        print("WARNING: GPU runtime but paddle is CPU build. Falling back to CPU paddle.")
        !pip uninstall -y paddlepaddle-gpu 2>&1 | tail -2
        %pip install -q paddlepaddle==3.0.0
        print("Restart the runtime (Runtime -> Restart session) to pick up the swap.")
except Exception as e:
    print(f"paddle import failed (expected on first install, restart runtime): {e}")


## Configure inputs and outputs

Point `PDF_PATH` at the SIP you want to process. Outputs go to `outputs/<sip_stem>/`.

In [ ]:
TARGET_DPI = 300

# Find a SIP PDF. Priority: explicit PDF_PATH override, then data/sips/,
# then any PDF at the project root (legacy location).
PDF_PATH = None  # set this by hand to force a specific file

if PDF_PATH is None:
    candidates = sorted((PROJECT_ROOT / "data" / "sips").glob("*.pdf"))
    if not candidates:
        candidates = sorted(PROJECT_ROOT.glob("*.pdf"))
    if candidates:
        PDF_PATH = candidates[0]
        if len(candidates) > 1:
            print(f"Multiple PDFs found, using first. All: {[p.name for p in candidates]}")

assert PDF_PATH and PDF_PATH.exists(), (
    f"No SIP PDF found. Drop one in {PROJECT_ROOT/'data'/'sips'}/ "
    f"or set PDF_PATH explicitly."
)
OUT_DIR = PROJECT_ROOT / "outputs" / PDF_PATH.stem
print(f"Input : {PDF_PATH}")
print(f"Output: {OUT_DIR}")

## Stages 1-5: Preprocessing

Render at target DPI, drop colored markup, Sauvola binarize, crop title blocks off the sides, save.

In [ ]:
from IPython.display import Image as IPyImage
from sip_extractor import preprocessing

prep = preprocessing.run(PDF_PATH, OUT_DIR, target_dpi=TARGET_DPI)
print(f"Crop x=[{prep.crop_x_min}, {prep.crop_x_max}]")
print(f"Binary    : {prep.binary_path}")
print(f"Preview   : {prep.binary_preview_path}")
IPyImage(prep.binary_preview_path)

## Stage 6: OCR

Tile-based PaddleOCR over the cleaned grayscale (NOT the binary). First run downloads ~few hundred MB of models, cached under `~/.paddleocr/` (or `PADDLE_HOME` on Colab).

In [ ]:
from sip_extractor import ocr

text_entities = ocr.run(prep.gray_cropped, OUT_DIR)
print(f"Detected {len(text_entities)} unique text regions")
IPyImage(str(OUT_DIR / "text_overlay_preview.png"))

## Stage 7: Text classification

Bucket each detection into `signal_id`, `track_circuit`, `km_marker`, `track_label`, `point_id`, `dimension`, or `note`. Rewrites `text.json` with the `category` field added.

In [ ]:
from sip_extractor import classify

classify.run(text_entities, OUT_DIR, gray_cropped=prep.gray_cropped)

counts = classify.category_counts(text_entities)
print("Category breakdown:")
for cat, n in counts.most_common():
    print(f"  {cat:15s}: {n}")

IPyImage(str(OUT_DIR / "text_categorized_preview.png"))

## Outputs

After this notebook runs, `outputs/<sip_stem>/` contains:

- `binary.png` — full-resolution cropped binary
- `binary_preview.png` — 3000px-wide preview
- `text.json` — detections with `text`, `text_normalized`, `score`, `bbox`, `category`
- `text_overlay.png` / `text_overlay_preview.png` — raw OCR overlay
- `text_categorized_overlay.png` / `text_categorized_preview.png` — color-coded by category

Stage 8 (track detection) is broken and deferred until after Stage 9 (symbol detection). See HANDOFF.md.